# R4 · 4 · Ensemble por regime, avaliação e gráficos

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem tudo das rodadas 1 a 3):
1. suba **`r4_lib.py`** (pasta `data_imdc_2026/RODADA4/`) e confira que `r3_lib.py` e `r2_lib.py` estão lá;
2. as pastas `resultados_uf_r4`, `revisoes_r4`, `cache_r4` e `avaliacao_r4` são criadas sozinhas.

**Usa a rodada 3:** o normal (ENS_grafos3_λ) vem de `cache_r3/normal_ens_grafos3_bb_clim.parquet` e as misturas da R3 entram na comparação.

**Complementar:** nada das rodadas anteriores muda. As previsões oficiais (corte de junho) vão para `resultados_uf_r4/uf/`; as revisões de julho a outubro, para `revisoes_r4/uf/`; as tabelas, para `avaliacao_r4/` (com data e hora no nome e uma cópia `_ultimo`).

**Retomada:** se a internet ou a luz cair, reconecte e **rode tudo de novo**. Cada etapa grava o seu arquivo (mar, sorotipo por ano, variáveis, detector por método × corte, parâmetros, um arquivo por modelo × temporada, os pesos das redes a cada 5 épocas, um PNG por UF) e o que já existe é pulado.

CPU comum, 15–30 min. **Rode depois do R4_2 e do R4_3.** Se o R4_3 ainda não terminou, roda com os membros que existirem e é só rodar de novo depois.

- **Membros:** normal (ENS_grafos3_λ), MIX_escala_UF e AJ_continuo_UF (R3), MIX_elastico_UF, DoisEstagios_det_UF, MTGNN_cond_UF e MTGNN_aug_UF.
- **Regimes:** a probabilidade de ano grande que o detector deu em junho define três faixas: baixa (< 0,35), média e alta (≥ 0,65).
- **`ENS_regime_UF`:** em cada faixa, os pesos dos membros (média dos quantis) são escolhidos na afinação (2018–2021) pelo **critério misto**. Assim, em ano tranquilo pesa mais o normal; em ano de alarme, pesam mais os modelos de pico.
- **`ENS_regime_UF+cal`:** por faixa, a largura e a posição da faixa de previsão são calibradas na afinação, que é a versão por faixas da calibração conforme. A posição mexe no máximo ×1,5.
- **Validação honesta antes do teste:** os dois ensembles são refeitos 4 vezes, cada vez deixando uma temporada da afinação de fora. A **recomendada** é o modelo de menor critério misto na afinação, contando os ensembles só fora da amostra. Ela é decidida antes de olhar o teste.
- **Régua no teste (2022–2025):**
  - R × BB (régua oficial), critério misto contra o BB e WIS absoluto;
  - fases (pico, intermediária, entressafra), erro na altura e na semana do pico, erro no total e cobertura;
  - as revisões mensais do MIX_elastico.
- **Gráficos:** 26 UFs + Brasil, com a recomendada, o normal e o BB, mais o que o detector disse em junho.


In [ ]:
NOTEBOOK = 'r4_avaliacao'

# @title 1 · Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/r4', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'uf_lib.py', 'semanal_lib.py', 'r2_lib.py', 'r3_lib.py',
                 'r4_lib.py', 'dados', 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    pass
sys.path.insert(0, PASTA_LOCAL)
PASTA_R4 = os.path.join(PASTA_DRIVE, 'resultados_uf_r4')
PASTA_REV4 = os.path.join(PASTA_DRIVE, 'revisoes_r4')
PASTA_C4 = os.path.join(PASTA_DRIVE, 'cache_r4')
PASTA_AV4 = os.path.join(PASTA_DRIVE, 'avaliacao_r4')
PASTA_R3 = os.path.join(PASTA_DRIVE, 'resultados_uf_r3')
PASTA_C3 = os.path.join(PASTA_DRIVE, 'cache_r3')
PASTA_AV2 = os.path.join(PASTA_DRIVE, 'avaliacao_r2')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
PASTA_EXT = os.path.join(PASTA_DRIVE, 'resultados_externos')
for p in (os.path.join(PASTA_R4, 'uf'), os.path.join(PASTA_REV4, 'uf'), PASTA_C4, os.path.join(PASTA_C4, 'ckpt'), PASTA_AV4):
    os.makedirs(p, exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U, semanal_lib as S, r2_lib as R2, r3_lib as R3, r4_lib as R4
for m in (V, L, CM, U, S, R2, R3, R4):
    importlib.reload(m)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
D = ctx.Du
TUNING, TESTE = U.TUNING, U.TESTE
REFAZER = False
R4.PESO_ABS = 0.5     # critério misto = (1 − p)·R × referência + p·WIS absoluto relativo. Suba (ex.: 0.7) para priorizar a altura dos picos


In [ ]:
# @title 2 · Ler tudo
NORMAL = R4.ler_normal(ctx, PASTA_DRIVE, PASTA_AV2, PASTA_COMP, PASTA_EXT, PASTA_C3, TUNING + TESTE)
R3M = R4.ler_pastas(PASTA_R3)
R3M = R3M[R3M.modelo.isin(['MIX_escala_UF', 'AJ_continuo_UF'])]
R4M = R4.ler_pastas(PASTA_R4)
R4M = R4M[~R4M.modelo.str.startswith('ENS_regime')]
REV = R4.ler_pastas(PASTA_REV4)
ESC, DET = R4.detector_escolhido4(PASTA_C4)
TODOS = pd.concat([NORMAL, R3M, R4M], ignore_index=True)
CAND = ['ENS_grafos3_λ', 'MIX_escala_UF', 'AJ_continuo_UF', 'MIX_elastico_UF', 'DoisEstagios_det_UF', 'MTGNN_cond_UF', 'MTGNN_aug_UF']
cnt = TODOS.groupby('modelo').season.nunique()
MEMBROS = [m for m in CAND if cnt.get(m, 0) == len(TUNING + TESTE)]
print('membros com as 8 temporadas:', MEMBROS, '\nfaltando:', [m for m in CAND if m not in MEMBROS])


In [ ]:
# @title 3 · Afinação (2018–2021): critério misto de cada modelo contra a climatologia
AF = R4.criterio(ctx, TODOS[TODOS.season.isin(TUNING) & TODOS.modelo.isin(MEMBROS + ['climatologia'])])
R4.salvar_tabela(AF, PASTA_AV4, 'afinacao_criterio_misto')
AF.round(3)


In [ ]:
# @title 4 · Ensemble por regime: validação na afinação (deixando uma temporada de fora), ajuste final e recomendada
XR = R4.com_regime(TODOS[TODOS.modelo.isin(MEMBROS + ['climatologia'])], DET)
CV = R4.cv_regime(ctx, XR, MEMBROS)                                   # fora da amostra dentro de 2018–2021
AFc = R4.criterio(ctx, pd.concat([TODOS[TODOS.season.isin(TUNING) & TODOS.modelo.isin(MEMBROS + ['climatologia'])], CV[L.COLS_PREV]]))
R4.salvar_tabela(AFc, PASTA_AV4, 'afinacao_criterio_misto_com_ensemble')
RECOMENDADA = AFc.modelo.iloc[0]                                      # decidida ANTES de olhar o teste
print('recomendada (menor critério misto na afinação; ensembles fora da amostra):', RECOMENDADA)
display(AFc.round(3))
PESOS, CAL = R4.ajustar_regime(ctx, XR[XR.season.isin(TUNING)], MEMBROS)   # ajuste final com as 4 temporadas
ENS, ENSc = R4.prever_regime(XR, MEMBROS, PESOS, CAL)
R4.json_grava(dict(membros=MEMBROS, pesos=PESOS, calibracao=CAL, faixas=list(R4.FAIXAS), recomendada=RECOMENDADA),
              os.path.join(PASTA_AV4, f'ensemble_regime_{R4.carimbo()}.json'))
for nome, x in (('ENS_regime_UF', ENS), ('ENS_regime_UF+cal', ENSc)):
    R4.grava_modelo(x[L.COLS_PREV], PASTA_R4, nome)
print('pesos por faixa de P(ano grande):'); display(pd.DataFrame(PESOS).round(2))
print('calibração por faixa (λ = largura, δ = deslocamento em log):'); pd.DataFrame(CAL).round(3)


In [ ]:
# @title 5 · Régua no TESTE (corte de junho): geral, critério misto, fases, pico, total e cobertura
TODOS = pd.concat([TODOS, ENS[L.COLS_PREV], ENSc[L.COLS_PREV]], ignore_index=True)
OF = TODOS[TODOS.season.isin(TESTE)]
_, RES = V.avaliar(OF[L.COLS_PREV], ctx.ver_uf, ref=('BB', 'climatologia'))
RES = RES[RES.temporadas == len(TESTE)]
CRIT = R4.criterio(ctx, OF, ref='BB')[['modelo', 'WIS_rel', 'criterio']].rename(columns={'WIS_rel': 'WIS_abs_rel_BB', 'criterio': 'criterio_misto_BB'})
TT = S.com_verdade(OF, D)
pont = pd.DataFrame([dict(modelo=m, **S.metricas(g)) for m, g in TT.groupby('modelo')])
LIM = R2.limiares(D)
FASE = R2.metricas_fase(R2.com_fase(TT, D, LIM), ['modelo'])
PICO = R2.erros_pico(TT, 'season')
COB = R2.cobertura_quantis(TT)
f1 = FASE.pivot_table(index='modelo', columns='fase', values=['WIS', 'WIS_log', 'vies_pct', 'IC90'])
f1.columns = [f'{a}_{b}' for a, b in f1.columns]
TAB = RES[['modelo', 'R_vs_BB', 'UFs_melhor_que_BB', 'R_vs_climatologia']].merge(CRIT, on='modelo', how='left') \
    .merge(pont, on='modelo').merge(f1.reset_index(), on='modelo', how='left').merge(PICO, on='modelo', how='left').sort_values('criterio_misto_BB')
for nome, t in (('temporada_teste_r4', TAB), ('temporada_teste_por_fase_r4', FASE), ('temporada_teste_pico_r4', PICO),
                ('temporada_teste_cobertura_r4', COB)):
    R4.salvar_tabela(t, PASTA_AV4, nome)
cols = ['modelo', 'R_vs_BB', 'WIS_abs_rel_BB', 'criterio_misto_BB', 'WIS', 'WAPE', 'vies_pct', 'IC90', 'WIS_alta (pico)', 'vies_pct_alta (pico)',
        'vies_pct_entressafra', 'IC90_alta (pico)', 'erro_altura_mediano_pct', 'erro_total_mediano_pct', 'pico_no_IC90_pct']
TAB[[c for c in cols if c in TAB]].round(3)


In [ ]:
# @title 6 · Revisões mensais do MIX_elastico (R × BB e erro no total por corte)
lin = []
for corte in R4.CORTES:
    nome = 'MIX_elastico_UF' if corte == 'jun' else f'MIX_elastico_UF@{corte}'
    g = pd.concat([TODOS, REV])[lambda x: (x.modelo == nome) & x.season.isin(TESTE)]
    if g.empty:
        continue
    _, r = V.avaliar(pd.concat([g.assign(modelo='x'), NORMAL[NORMAL.modelo == 'BB']])[L.COLS_PREV], ctx.ver_uf, ref=('BB',))
    pk = R2.erros_pico(S.com_verdade(g.assign(modelo='x'), D), 'season')
    lin.append(dict(corte=corte, R_vs_BB=float(r.set_index('modelo').loc['x', 'R_vs_BB']),
                    erro_altura_mediano_pct=float(pk.erro_altura_mediano_pct.iloc[0]),
                    erro_total_mediano_pct=float(pk.erro_total_mediano_pct.iloc[0]), pico_no_IC90_pct=float(pk.pico_no_IC90_pct.iloc[0])))
REVT = pd.DataFrame(lin)
R4.salvar_tabela(REVT, PASTA_AV4, 'revisoes_mensais_r4')
REVT.round(3)


In [ ]:
# @title 7 · Gráficos (26 UFs + Brasil), PDF e totais por temporada
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import Image, display
PASTA_G = os.path.join(PASTA_AV4, f'graficos_r4_{R4.carimbo()[:8]}')
os.makedirs(PASTA_G, exist_ok=True)
OBS = pd.DataFrame(D.Y[:D.ultimo_obs + 1], index=pd.to_datetime(D.datas[:D.ultimo_obs + 1]), columns=D.unidades)
OBS[0] = OBS.sum(1)
PREV = {f'recomendada · {RECOMENDADA}': TODOS[TODOS.modelo == RECOMENDADA],
        'normal · ENS_grafos3_λ': TODOS[TODOS.modelo == 'ENS_grafos3_λ'],
        'modelo-base oficial · BB': TODOS[TODOS.modelo == 'BB']}
DJ = DET[DET.corte == 'jun']
ORDEM = [0] + sorted(D.unidades, key=lambda u: R2.SIGLAS.get(u, str(u)))
tot = []
for u in ORDEM:
    arq = os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')
    if not os.path.exists(arq) or REFAZER:
        det = DJ[DJ.uf_code == u].set_index('season') if u != 0 else None
        fig = R3.grafico_temporada_uf(OBS[u], PREV, u, arq, TESTE, det=det,
                                      titulo=' · rodada 4' + (' · Brasil = soma das UFs' if u == 0 else ''))
        plt.close(fig)
    tot.append(R2.totais_temporada(OBS[u], PREV, u, TESTE))
with PdfPages(os.path.join(PASTA_G, 'graficos_UFs_r4.pdf.tmp')) as pdf:
    for u in ORDEM:
        img = plt.imread(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png'))
        f = plt.figure(figsize=(13, 4.6)); plt.imshow(img); plt.axis('off'); pdf.savefig(f, bbox_inches='tight'); plt.close(f)
os.replace(os.path.join(PASTA_G, 'graficos_UFs_r4.pdf.tmp'), os.path.join(PASTA_G, 'graficos_UFs_r4.pdf'))
TOT = pd.concat(tot, ignore_index=True)
TOT.to_csv(os.path.join(PASTA_G, 'totais_temporada_r4.csv'), index=False)
print('Pasta:', PASTA_G)
for u in (0, 35, 31, 41, 42, 53):
    display(Image(os.path.join(PASTA_G, f'{R2.SIGLAS.get(u, u)}.png')))
TOT.round(0)
